Start from the dataset kickstarter-14-04, take out the unnecessary columsn that I don't know why are there:

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import nltk
from nltk.corpus import stopwords
import string 
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
import re 
from collections import Counter

take out the unnecessary columns, add the reached one and classify as success or failure (maybe it's already in the initial dataset but honestly I don't remember)

In [2]:
campaigns = pd.read_csv('Kickstarter_dataset_14-04.csv')
campaigns = campaigns.drop(columns = ['Unnamed: 0', 'description_processed', 'stemmed', 'lemmatized'])
campaigns['reached'] = (campaigns['pledged'] / campaigns['goal']) * 100

median_reached = campaigns['reached'].median()
campaigns['status'] = campaigns['reached'].apply(lambda x: 1 if x >= median_reached else 0)

Preprocessing: usual preprocessing stuff like lowercasing, taking out links, only keepin alpha numeric characters, tokenizing the words, and taking out the english stopwords, also took out the words that are less than 2 characters

In [3]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text_p = "".join([char for char in text if char.isalnum() or char.isspace()])
    
    words = word_tokenize(text_p) 
    
    
    stop_words = stopwords.words('english')
    words = [w for w in words if w not in stop_words and len(w) > 2]
    filtered_words = [word for word in words if word not in stop_words] 
    
    return filtered_words  

campaigns['description_processed'] = campaigns['description'].apply(preprocess)

How do we decide which stopwords to include beside the 'normal' stopwords? We also want to include words that are very generic, not informative at all and coudl potentially appear in any kind of campaign, regardless of the topics in it (some examples could be the words 'kickstarter', campaign, etc etc)
Initially I thought TF-IDF would make sense but it doesn't actually, because TF-IDF both depends on how frequent a word is in each document and how frequent are documents that contain that words in the whole set of documents. (also you cannot average the TF-IDF of a word and pick the ones with lowest value because a rare word, which we wanna keep, could end up having a low TF-IDF value if it appears in few documents but many time in those few documents)
So potentially we can just use document frequency, meaning if the word appears in > alpha% of documents, we count it as a stopword?

Also quick sidenote: Other methods like BytePair encoding, WordPiece or SentencePiece are not really useful here because they look at subwords, and for example if we get 'kickstarter' and split it into kick and starter, then we might consider to keep kick because it makes sense with sport but we might not have any document where the word is present for real

This below just basically gives a counter of the document frequency of each token in our whole corpus of all the descriptions 

In [43]:
docs = campaigns['lemmatized']
N = len(docs)

df_counter = Counter()

for doc in docs:
    df_counter.update(set(doc))

df_table = pd.DataFrame({
    'word': list(df_counter.keys()),
    'doc_freq': list(df_counter.values())
})

df_table['doc_freq_ratio'] = df_table['doc_freq'] / N
df_table = df_table.sort_values('doc_freq_ratio', ascending=False)

print(df_counter)
df_table

Counter({'one': 5194, 'make': 5190, 'time': 5127, 'help': 4925, 'project': 4743, 'new': 4693, 'like': 4692, 'also': 4677, 'year': 4663, 'first': 4513, 'get': 4459, 'need': 4438, 'world': 4395, 'work': 4373, 'life': 4237, 'way': 4180, 'support': 4048, 'kickstarter': 4005, 'goal': 3952, 'well': 3860, 'want': 3813, 'people': 3717, 'story': 3677, 'many': 3661, 'take': 3584, 'every': 3567, 'come': 3532, 'see': 3522, 'love': 3516, 'experience': 3475, 'even': 3376, 'campaign': 3274, 'two': 3178, 'back': 3157, 'create': 3135, 'part': 3132, 'production': 3112, 'reward': 3085, 'design': 3078, 'bring': 3069, 'team': 3017, 'making': 3007, 'made': 3006, 'know': 2996, 'friend': 2996, 'much': 2977, 'best': 2969, 'would': 2962, 'set': 2906, 'day': 2904, 'together': 2898, 'cost': 2840, 'use': 2752, 'feature': 2750, 'video': 2741, 'around': 2722, 'right': 2676, 'art': 2673, 'music': 2633, 'find': 2620, 'thing': 2618, 'play': 2590, 'working': 2584, 'community': 2575, 'thank': 2573, 'show': 2572, 'give': 

,word,doc_freq,doc_freq_ratio
261,one,5194,0.706282
297,make,5190,0.705738
323,time,5127,0.697172
250,help,4925,0.669704
143,project,4743,0.644955
...,...,...,...
120177,unworn,1,0.000136
120176,jennyshopatsoracom,1,0.000136
120175,estimator,1,0.000136
120174,sundaze,1,0.000136


Now the only method and the one that made most sense to me to take out too common words is basically just defining a threshold and see which words are too rare (like if it appears only 2-3 times in the whole dataset) or too much (in this case too much can be like appearing in more than 40-60% of the descriptions), you can try different values of the two initial thresholds (the one I saw looks the best would be 0.0005 for the low and 0.55 for the high)

In [45]:
min_ratio = 0.0004
max_ratio = 0.50

vocab = set(
    df_table[
        (df_table['doc_freq_ratio'] > min_ratio) &
        (df_table['doc_freq_ratio'] < max_ratio)
    ]['word']
)

campaigns['description_filtered'] = campaigns['lemmatized'].apply(
    lambda doc: [w for w in doc if w in vocab]
)

print(df_table[df_table['doc_freq_ratio'] >= max_ratio])
print(df_table[df_table['doc_freq_ratio'] <= min_ratio])

             word  doc_freq  doc_freq_ratio
261           one      5194        0.706282
297          make      5190        0.705738
323          time      5127        0.697172
250          help      4925        0.669704
143       project      4743        0.644955
1080          new      4693        0.638156
36           like      4692        0.638020
80           also      4677        0.635980
883          year      4663        0.634077
231         first      4513        0.613680
427           get      4459        0.606337
706          need      4438        0.603481
236         world      4395        0.597634
844          work      4373        0.594642
242          life      4237        0.576149
163           way      4180        0.568398
367       support      4048        0.550449
391   kickstarter      4005        0.544602
376          goal      3952        0.537395
1034         well      3860        0.524884
776          want      3813        0.518493
283        people      3717     

ONLY RUN THE NEXT CELL IF YOU WANT TO ACTUALLY REMOVE THOSE TOKENS THAT ARE BEYOND THE THRESHOLDS YOU SET 


In [ ]:
campaigns['description_processed'] = campaigns['description_processed'].apply(
    lambda doc: [w for w in doc if w.isalpha() and len(w) > 2]
)
campaigns['description_processed']

then you can apply standard stemming / lemmatization: 


In [38]:
porter = PorterStemmer()
lemmatizer = WordNetLemmatizer()

campaigns['stemmed'] = campaigns['description_processed'].apply(
    lambda words: [porter.stem(word) for word in words]
)

campaigns['lemmatized'] = campaigns['description_processed'].apply(
    lambda words: [lemmatizer.lemmatize(word) for word in words])

In [ ]:

for cat, subset in campaigns.groupby('category'):
    
    counter = Counter(
        word for doc in subset['stemmed'] for word in doc
    )
    
    print(f"\nCategory: {cat}")
    print(counter.most_common(20))



Category: Film & Video
[('film', 18842), ('work', 5768), ('stori', 5631), ('make', 5625), ('product', 4857), ('produc', 4624), ('project', 4594), ('help', 4445), ('one', 3963), ('new', 3777), ('get', 3664), ('year', 3656), ('time', 3484), ('also', 3443), ('like', 3432), ('world', 3418), ('life', 3086), ('festiv', 3086), ('support', 3048), ('includ', 3013)]

Category: Games
[('game', 17660), ('play', 6256), ('card', 6082), ('player', 4949), ('new', 4628), ('design', 4515), ('make', 4287), ('ship', 4268), ('one', 4215), ('world', 4170), ('use', 4044), ('includ', 3767), ('get', 3694), ('charact', 3532), ('creat', 3454), ('kickstart', 3384), ('time', 3319), ('like', 3293), ('goal', 3260), ('work', 3125)]

Category: Music
[('music', 6197), ('album', 5042), ('record', 4812), ('song', 3996), ('project', 2552), ('make', 2492), ('help', 2324), ('new', 2282), ('time', 2181), ('year', 2130), ('work', 2083), ('one', 1859), ('support', 1805), ('get', 1711), ('love', 1656), ('studio', 1638), ('rele